In [1]:
!pip install -q --upgrade \
    "transformers==4.57.6" \
    "tokenizers>=0.22.0,<=0.23.0" \
    "trl==0.22.2" \
    unsloth unsloth_zoo

!pip install -q --no-deps --upgrade "torchao>=0.16.0"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
import unsloth

from unsloth import FastLanguageModel
import torch
import transformers
import trl

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("CUDA:", torch.version.cuda)
print("BF16 supported:", torch.cuda.is_bf16_supported())

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
PyTorch: 2.11.0+cu128
Transformers: 4.57.6
TRL: 0.22.2
CUDA: 12.8
BF16 supported: False


In [3]:
import unsloth
import torch
import transformers
import trl
import unsloth as us

print("Unsloth:", us.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TRL:", trl.__version__)
print("GPU count:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

Unsloth: 2026.11.2
PyTorch: 2.11.0+cu128
Transformers: 4.57.6
TRL: 0.22.2
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


In [4]:
import os
import json
import torch

# ==========================================
# 1. FIXED MODEL AND DATASET PATHS
# ==========================================


TRAIN_FILE = (
    "/kaggle/input/datasets/devkanthravi/"
    "baseline-lora-dataset-oncology/"
    "final_QA_train_dataset_NCL.jsonl"
)

TEST_FILE = (
    "/kaggle/input/datasets/devkanthravi/manual-updated-testfile-oncology/generated_pairs_cleaned_exact.jsonl"
)

OUTPUT_DIR = "/kaggle/working/gpt_oss_20b_oncology_lora"

# ==========================================
# 2. CHECK ENVIRONMENT
# ==========================================

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

assert torch.cuda.is_available(), "Enable GPU acceleration in Kaggle."

for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)

    print(f"\nGPU {i}: {props.name}")
    print(f"Total VRAM: {total / 1024**3:.2f} GiB")
    print(f"Free VRAM:  {free / 1024**3:.2f} GiB")

# ==========================================
# 3. VERIFY PATHS
# ==========================================
print("Training file exists:", os.path.isfile(TRAIN_FILE))
print("Test file exists:", os.path.isfile(TEST_FILE))


assert os.path.isfile(TRAIN_FILE), (
    f"Training dataset not found: {TRAIN_FILE}"
)
assert os.path.isfile(TEST_FILE), (
    f"Test dataset not found: {TEST_FILE}"
)

assert os.path.abspath(TRAIN_FILE) != os.path.abspath(TEST_FILE), (
    "Training and test files must be different."
)

# ==========================================
# 4. INSPECT TRAINING DATA
# ==========================================

train_data = []

with open(TRAIN_FILE, "r", encoding="utf-8") as f:
    for line_number, line in enumerate(f, start=1):
        if not line.strip():
            continue

        try:
            item = json.loads(line)
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Invalid JSON on line {line_number}: {e}"
            ) from e

        train_data.append(item)

print("\nTraining examples:", len(train_data))

assert len(train_data) > 0, "Training dataset is empty."

print("\nFirst training example:")
print(json.dumps(train_data[0], indent=2, ensure_ascii=False))

print("\nAvailable fields:", list(train_data[0].keys()))

print("\nCell 1 checks completed.")

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU count: 2

GPU 0: Tesla T4
Total VRAM: 14.56 GiB
Free VRAM:  14.41 GiB

GPU 1: Tesla T4
Total VRAM: 14.56 GiB
Free VRAM:  14.46 GiB
Training file exists: True
Test file exists: True

Training examples: 2537

First training example:
{
  "input": "What is cancer in terms of its biological definition?",
  "output": "Cancer is a disease in which some of the body’s cells grow uncontrollably and spread to other parts of the body. It is caused by changes to genes that control cell growth and division, leading to malfunctioning cells that do not die when they should or grow when they should not."
}

Available fields: ['input', 'output']

Cell 1 checks completed.


In [5]:
!pip install -q bert-score



In [ ]:

# ============================================================
# GPT-OSS-20B ONCOLOGY LoRA SFT — FRESH, GUARDED TRAINING RUN
# Kaggle: 2x Tesla T4
#
# Base model: exact same checkpoint used for baseline evaluation
# Training: 2537 examples
# Held-out evaluation: 102 examples
# Evaluation: BERTScore after every epoch
# ============================================================

import os
import gc
import json
import math
import inspect
import random
import numpy as np
import torch

from datasets import Dataset
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from bert_score import score


# ============================================================
# 1. CONFIGURATION
# ============================================================

MODEL_PATH = "unsloth/gpt-oss-20b-BF16"

TRAIN_FILE = (
    "/kaggle/input/datasets/devkanthravi/"
    "baseline-lora-dataset-oncology/"
    "final_QA_train_dataset_NCL.jsonl"
)

TEST_FILE = (
    "/kaggle/input/datasets/devkanthravi/manual-updated-testfile-oncology/generated_pairs_cleaned_exact.jsonl"
)

# A new directory prevents reuse of checkpoints from the failed run.
OUTPUT_DIR = "/kaggle/working/gpt_oss_20b_oncology_lora_stable_v2"
CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
FINAL_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "final_adapter")
PREDICTIONS_DIR = os.path.join(OUTPUT_DIR, "epoch_predictions")
METRICS_FILE = os.path.join(OUTPUT_DIR, "epoch_bertscore_results.json")

MAX_SEQ_LENGTH = 1024
MAX_NEW_TOKENS = 96
REASONING_EFFORT = "low"

NUM_EPOCHS = 3
LORA_RANK = 8
LORA_ALPHA = 16

# Conservative starting point after the previous run produced NaN.
LEARNING_RATE = 2e-4
WEIGHT_DECAY = 0.0
MAX_GRAD_NORM = 0.3

BASELINE_METRICS = {
    "precision": 0.8410430550575256,
    "recall": 0.8884621262550354,
    "f1": 0.8638684153556824,
}

SEED = 3407

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

# Reproducibility.
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. This run expects Kaggle GPUs.")


if not os.path.isfile(TRAIN_FILE):
    raise FileNotFoundError(f"Training file not found: {TRAIN_FILE}")

if not os.path.isfile(TEST_FILE):
    raise FileNotFoundError(f"Test file not found: {TEST_FILE}")

if os.path.abspath(TRAIN_FILE) == os.path.abspath(TEST_FILE):
    raise ValueError("Training and test files must be different.")

# Release any old notebook variables that could retain the previous
# model, LoRA adapter, trainer, or callbacks in memory.
for variable_name in [
    "trainer",
    "model",
    "tokenizer",
    "training_args",
    "train_dataset",
    "bertscore_callback",
    "train_result",
    "train_data",
    "train_on_responses_only",
]:
    globals().pop(variable_name, None)

gc.collect()
torch.cuda.empty_cache()

print("Model checkpoint:", MODEL_PATH)
print("Training file:", TRAIN_FILE)
print("Held-out test file:", TEST_FILE)
print("Visible GPUs:", torch.cuda.device_count())
for gpu_index in range(torch.cuda.device_count()):
    print(
        f"GPU {gpu_index}: "
        f"{torch.cuda.get_device_name(gpu_index)}"
    )


# ============================================================
# 2. LOAD AND VALIDATE THE TRAINING DATA
# ============================================================

training_rows = []

with open(TRAIN_FILE, "r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue

        try:
            item = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid training JSON at line {line_number}"
            ) from error

        question = item.get("input")
        answer = item.get("output")

        if not isinstance(question, str) or not isinstance(answer, str):
            raise ValueError(
                f"Invalid input/output fields at training line {line_number}"
            )

        question = question.strip()
        answer = answer.strip()

        if not question or not answer:
            continue

        # Conversational prompt-completion format.
        # TRL applies the tokenizer's chat template.
        training_rows.append({
            "prompt": [
                {"role": "user", "content": question}
            ],
            "completion": [
                {"role": "assistant", "content": answer}
            ],
        })

if not training_rows:
    raise ValueError("No valid training examples were found.")

train_dataset = Dataset.from_list(training_rows)

print("\nTraining examples:", len(train_dataset))
print("Training format:", train_dataset.features)
print("Example prompt:", train_dataset[0]["prompt"])
print("Example completion:", train_dataset[0]["completion"])


# ============================================================
# 3. LOAD THE HELD-OUT TEST DATA
# ============================================================

questions = []
references = []

with open(TEST_FILE, "r", encoding="utf-8") as file:
    for line_number, line in enumerate(file, start=1):
        if not line.strip():
            continue

        try:
            item = json.loads(line)
        except json.JSONDecodeError as error:
            raise ValueError(
                f"Invalid test JSON at line {line_number}"
            ) from error

        question = item.get("input")
        reference = item.get("output")

        if not isinstance(question, str) or not isinstance(reference, str):
            raise ValueError(
                f"Invalid input/output fields at test line {line_number}"
            )

        question = question.strip()
        reference = reference.strip()

        if question and reference:
            questions.append(question)
            references.append(reference)

if not questions:
    raise ValueError("No valid held-out test examples were found.")

if len(questions) != len(references):
    raise AssertionError("Test question/reference counts do not match.")

print("\nTraining examples:", len(train_dataset))
print("Held-out test examples:", len(questions))


# ============================================================
# 4. LOAD THE NF4 BASE MODEL
# ============================================================

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_PATH,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.bfloat16,
    load_in_4bit=False,
    full_finetuning=False,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model.config.use_cache = False

print("\nNF4 base model loaded.")
print("Model:", MODEL_PATH)
print("Quantization config:", model.config.quantization_config)

# ============================================================
# 5. ATTACH FRESH LoRA ADAPTERS
# ============================================================

# Keep the GPT-OSS attention targets used in the previous setup.
# Do not assume standard MLP target names work for this architecture.

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

if trainable_parameters == 0:
    raise RuntimeError("No trainable LoRA parameters were found.")

print("\nFresh LoRA adapter attached.")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Total parameters:     {total_parameters:,}")
print(
    f"Trainable percentage: "
    f"{100 * trainable_parameters / total_parameters:.5f}%"
)


# ============================================================
# 6. TRAINING CONFIGURATION
# ============================================================

training_args = SFTConfig(
    output_dir=CHECKPOINT_DIR,

    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=LEARNING_RATE,
    lr_scheduler_type="linear",
    warmup_ratio=0.03,
    weight_decay=WEIGHT_DECAY,
    max_grad_norm=MAX_GRAD_NORM,

    optim="adamw_8bit",

    logging_strategy="steps",
    logging_steps=5,
    logging_first_step=True,

    # Save checkpoints during training, not only at epoch boundaries.
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,

    gradient_checkpointing=True,

    # Conversational prompt-completion examples: compute loss on
    # completion tokens, not the user's prompt.
    completion_only_loss=True,

    max_length=MAX_SEQ_LENGTH,
    packing=False,

    report_to="none",
    seed=SEED,
    data_seed=SEED,
)


# ============================================================
# 7. FAIL FAST ON NON-FINITE LOSS
# ============================================================

class FiniteLossSFTTrainer(SFTTrainer):
    """
    Stop before backpropagation/optimizer update if the computed loss
    is NaN or infinite.
    """

    def compute_loss(
        self,
        model,
        inputs,
        return_outputs=False,
        num_items_in_batch=None,
        **kwargs,
    ):
        parent_compute_loss = super().compute_loss
        parent_parameters = inspect.signature(
            parent_compute_loss
        ).parameters

        parent_kwargs = {}

        if "return_outputs" in parent_parameters:
            parent_kwargs["return_outputs"] = return_outputs

        if (
            "num_items_in_batch" in parent_parameters
            and num_items_in_batch is not None
        ):
            parent_kwargs["num_items_in_batch"] = num_items_in_batch

        for key, value in kwargs.items():
            if key in parent_parameters:
                parent_kwargs[key] = value

        result = parent_compute_loss(
            model,
            inputs,
            **parent_kwargs,
        )

        loss = result[0] if isinstance(result, tuple) else result

        if not torch.isfinite(loss.detach()).all().item():
            raise FloatingPointError(
                "Non-finite training loss detected. "
                "Training stopped before this loss could be backpropagated. "
                "Do not use this run's adapter as a completed model."
            )

        return result


class FiniteGradientCallback(TrainerCallback):
    """
    Guard trainable gradients before the optimizer step.
    """

    def on_pre_optimizer_step(self, args, state, control, **kwargs):
        current_model = kwargs.get("model")

        if current_model is None:
            return control

        for name, parameter in current_model.named_parameters():
            if (
                parameter.requires_grad
                and parameter.grad is not None
                and not torch.isfinite(parameter.grad).all().item()
            ):
                raise FloatingPointError(
                    f"Non-finite gradient detected in {name} "
                    f"at optimizer step {state.global_step}. "
                    "Training stopped before the optimizer step."
                )

        return control


trainer = FiniteLossSFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

# Verify that completion-only loss is actually supervising examples.
# The trainer tokenizes the dataset during initialization.

# ============================================================
# VERIFY TOKENIZATION AND COMPLETION-ONLY LOSS
# Compatible with the TRL 0.22.2 pipeline in this notebook
# ============================================================

processed_dataset = trainer.train_dataset

if len(processed_dataset) == 0:
    raise RuntimeError("The processed training dataset is empty.")

required_columns = {"input_ids", "completion_mask"}
missing_columns = required_columns - set(processed_dataset.column_names)

if missing_columns:
    raise RuntimeError(
        f"Missing required tokenized fields: {sorted(missing_columns)}. "
        f"Available fields: {processed_dataset.column_names}"
    )

supervised_example_count = 0

for index in range(len(processed_dataset)):
    example = processed_dataset[index]
    input_ids = example["input_ids"]
    completion_mask = example["completion_mask"]

    if len(input_ids) != len(completion_mask):
        raise RuntimeError(
            f"Token/mask length mismatch at example {index}: "
            f"{len(input_ids)} tokens vs {len(completion_mask)} mask entries."
        )

    if len(input_ids) > MAX_SEQ_LENGTH:
        raise RuntimeError(
            f"Example {index} exceeds MAX_SEQ_LENGTH: {len(input_ids)}."
        )

    if any(int(value) == 1 for value in completion_mask):
        supervised_example_count += 1

if supervised_example_count != len(processed_dataset):
    raise RuntimeError(
        f"Only {supervised_example_count}/{len(processed_dataset)} "
        "examples contain supervised completion tokens."
    )

# Test the actual collator: labels should exist in its output,
# and prompt tokens should be masked with -100.
sample_examples = [
    processed_dataset[i]
    for i in range(min(2, len(processed_dataset)))
]

batch = trainer.data_collator(sample_examples)

if "labels" not in batch:
    raise RuntimeError(
        "The training data collator did not produce labels."
    )

for row_index, example in enumerate(sample_examples):
    completion_mask = torch.tensor(
        example["completion_mask"], dtype=torch.bool
    )
    labels = batch["labels"][row_index][:len(completion_mask)]

    actual_supervision = labels.ne(-100)

    if not torch.equal(actual_supervision.cpu(), completion_mask):
        raise RuntimeError(
            f"Completion-only label masking failed for sample {row_index}."
        )

print("\nTrainer created and tokenization checks passed.")
print("Processed dataset columns:", processed_dataset.column_names)
print(
    "Examples with supervised completion tokens:",
    supervised_example_count,
    "/",
    len(processed_dataset),
)
print("Collator verified: prompt tokens masked, completion tokens supervised.")

del sample_examples, batch

trainer.add_callback(FiniteGradientCallback())


# ============================================================
# 8. EPOCH-END GENERATION AND BERTSCORE EVALUATION
# ============================================================

class EpochBERTScoreCallback(TrainerCallback):

    def __init__(self, model, tokenizer, questions, references):
        self.model_ref = model
        self.tokenizer = tokenizer
        self.questions = questions
        self.references = references
        self.results = []

    def on_epoch_end(self, args, state, control, **kwargs):
        epoch = int(round(state.epoch or 0))

        print("\n" + "=" * 20)
        print(f"EPOCH {epoch}: HELD-OUT EVALUATION")
        print("=" * 20)

        eval_model = self.model_ref
        predictions = []

        prediction_file = os.path.join(
            PREDICTIONS_DIR,
            f"epoch_{epoch}_predictions.jsonl",
        )

        FastLanguageModel.for_inference(eval_model)

        try:
            eval_model.eval()

            input_device = (
                eval_model.get_input_embeddings().weight.device
            )

            # Write each result immediately so progress survives
            # an interruption during long inference.
            with open(
                prediction_file,
                "w",
                encoding="utf-8",
            ) as prediction_handle:

                for index, question in enumerate(self.questions):
                    # Match the original baseline evaluation prompt.
                    messages = [{
                        "role": "user",
                        "content": (
                            "Answer the following general cancer-knowledge "
                            "question clearly and concisely. Give a direct "
                            "answer without unnecessary reasoning.\n\n"
                            f"Question: {question}"
                        ),
                    }]

                    inputs = self.tokenizer.apply_chat_template(
                        messages,
                        tokenize=True,
                        add_generation_prompt=True,
                        return_tensors="pt",
                        return_dict=True,
                        reasoning_effort=REASONING_EFFORT,
                    )

                    inputs = {
                        key: value.to(input_device)
                        for key, value in inputs.items()
                    }

                    prompt_length = inputs["input_ids"].shape[1]

                    with torch.inference_mode():
                        output_ids = eval_model.generate(
                            **inputs,
                            max_new_tokens=MAX_NEW_TOKENS,
                            do_sample=False,
                            pad_token_id=self.tokenizer.eos_token_id,
                        )

                    generated_ids = output_ids[0, prompt_length:]

                    prediction = self.tokenizer.decode(
                        generated_ids,
                        skip_special_tokens=True,
                    ).strip()

                    predictions.append(prediction)

                    prediction_handle.write(
                        json.dumps(
                            {
                                "input": question,
                                "reference": self.references[index],
                                "prediction": prediction,
                            },
                            ensure_ascii=False,
                        ) + "\n"
                    )
                    prediction_handle.flush()

                    del inputs, output_ids, generated_ids

                    if (
                        (index + 1) % 10 == 0
                        or index + 1 == len(self.questions)
                    ):
                        print(
                            f"Generated {index + 1}/"
                            f"{len(self.questions)} answers"
                        )

        finally:
            FastLanguageModel.for_training(eval_model)
            gc.collect()
            torch.cuda.empty_cache()

        if len(predictions) != len(self.references):
            raise RuntimeError(
                "Prediction/reference count mismatch; "
                "BERTScore evaluation aborted."
            )

        print("\nCalculating BERTScore on CPU to avoid competing "
              "with GPT-OSS for GPU memory.")

        # Same metric encoder and scoring definition as the baseline.
        # CPU execution avoids loading RoBERTa-large onto an already
        # heavily occupied training GPU.
        P, R, F1 = score(
            cands=predictions,
            refs=self.references,
            model_type="roberta-large",
            lang="en",
            batch_size=8,
            device="cpu",
            rescale_with_baseline=False,
            verbose=True,
        )

        precision = P.mean().item()
        recall = R.mean().item()
        f1 = F1.mean().item()

        result = {
            "epoch": epoch,
            "global_step": int(state.global_step),
            "model": "GPT-OSS-20B",
            "fine_tuned": True,
            "metric": "BERTScore",
            "encoder": "roberta-large",
            "rescale_with_baseline": False,
            "scoring_device": "cpu",
            "num_test_examples": len(self.questions),
            "max_new_tokens": MAX_NEW_TOKENS,
            "reasoning_effort": REASONING_EFFORT,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "baseline_precision": BASELINE_METRICS["precision"],
            "baseline_recall": BASELINE_METRICS["recall"],
            "baseline_f1": BASELINE_METRICS["f1"],
            "precision_delta_from_baseline": (
                precision - BASELINE_METRICS["precision"]
            ),
            "recall_delta_from_baseline": (
                recall - BASELINE_METRICS["recall"]
            ),
            "f1_delta_from_baseline": (
                f1 - BASELINE_METRICS["f1"]
            ),
        }

        self.results.append(result)

        print(f"\nEpoch {epoch} BERTScore")
        print(f"Precision: {precision:.6f}")
        print(f"Recall:    {recall:.6f}")
        print(f"F1:        {f1:.6f}")
        print(
            f"F1 change vs baseline: "
            f"{result['f1_delta_from_baseline']:+.6f}"
        )

        with open(METRICS_FILE, "w", encoding="utf-8") as file:
            json.dump(self.results, file, indent=2)

        print("Metrics saved:", METRICS_FILE)
        print("Predictions saved:", prediction_file)

        del P, R, F1, predictions
        gc.collect()
        return control


bertscore_callback = EpochBERTScoreCallback(
    model=model,
    tokenizer=tokenizer,
    questions=questions,
    references=references,
)

trainer.add_callback(bertscore_callback)


# ============================================================
# 9. TRAIN
# ============================================================

print("\nStarting fresh GPT-OSS-20B oncology SFT.")
print("Epochs:", NUM_EPOCHS)
print("Learning rate:", LEARNING_RATE)
print("LoRA rank:", LORA_RANK)
print("Batch size per device:", 1)
print("Gradient accumulation:", 8)
print("Maximum sequence length:", MAX_SEQ_LENGTH)
print("Gradient clipping:", MAX_GRAD_NORM)
print("Optimizer:", "adamw_8bit")
print("Checkpoint interval:", "100 optimizer steps")
print("BERTScore evaluation:", "after each epoch")

# No automatic resume from the previous failed run.
# If loss/gradients become non-finite, the guard raises an exception.
train_result = trainer.train(resume_from_checkpoint=False)


# ============================================================
# 10. SAVE THE FINAL ADAPTER ONLY AFTER SUCCESSFUL TRAINING
# ============================================================

os.makedirs(FINAL_ADAPTER_DIR, exist_ok=True)

model.save_pretrained(FINAL_ADAPTER_DIR)
tokenizer.save_pretrained(FINAL_ADAPTER_DIR)

print("\nTraining completed successfully.")
print("Final LoRA adapter:", FINAL_ADAPTER_DIR)
print("Metrics:", METRICS_FILE)
print("Predictions directory:", PREDICTIONS_DIR)
print("Final training metrics:")
print(train_result.metrics)


Model checkpoint: unsloth/gpt-oss-20b-unsloth-bnb-4bit
Training file: /kaggle/input/datasets/devkanthravi/baseline-lora-dataset-oncology/final_QA_train_dataset_NCL.jsonl
Held-out test file: /kaggle/input/datasets/devkanthravi/manual-updated-testfile-oncology/generated_pairs_cleaned_exact.jsonl
Visible GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

Training examples: 2537
Training format: {'prompt': List({'content': Value('string'), 'role': Value('string')}), 'completion': List({'content': Value('string'), 'role': Value('string')})}
Example prompt: [{'content': 'What is cancer in terms of its biological definition?', 'role': 'user'}]
Example completion: [{'content': 'Cancer is a disease in which some of the body’s cells grow uncontrollably and spread to other parts of the body. It is caused by changes to genes that control cell growth and division, leading to malfunctioning cells that do not die when they should or grow when they should not.', 'role': 'assistant'}]

Training examples: 2537
He

Parse safetensors files:   0%|          | 0/4 [00:00<?, ?it/s]

total weights      : 11.361 GiB
no_split classes   : ['GptOssDecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.394 GiB
activation reserve : 7.116 GiB requested
  cuda:0  budget  12.97 GiB  weights  5.680 GiB  free  7.291 GiB  reserve  7.116 GiB
  cuda:1  budget  13.01 GiB  weights  5.680 GiB  free  7.334 GiB  reserve  6.940 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.413 -> 12.972 GiB, cuda:1 14.460 -> 13.014 GiB (memory the quantiser keeps for its own load-time buffers)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Unsloth: the model has bfloat16 layers on GPU 0 (Tesla T4), GPU 1 (Tesla T4), which has no native bfloat16, so they run slowly. Load with dtype = torch.float16, or keep the model on bfloat16 GPUs with CUDA_VISIBLE_DEVICES.

NF4 base model loaded.
Model: unsloth/gpt-oss-20b-unsloth-bnb-4bit
Quantization config: {'bnb_4bit_compute_dtype': torch.float16, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_use_double_quant': True, 'llm_int8_enable_fp32_cpu_offload': False, 'llm_int8_has_fp16_weight': False, 'llm_int8_skip_modules': ['router', 'lm_head', 'embed_tokens'], 'llm_int8_threshold': 6.0, 'load_in_4bit': True, 'load_in_8bit': False, 'quant_method': 'bitsandbytes'}
Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.

Fresh LoRA adapter attached.
Trainable parameters: 3,981,312
Total parameters:     11,045,084,736
Trainable percentage: 0.03605%
Unsloth: Switching to float32 training since model cannot w

Unsloth: Tokenizing ["prompt"+"completion"] (num_proc=2):   0%|          | 0/2537 [00:00<?, ? examples/s]

... (25 outputs left out)

  0%|          | 0/21 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/11 [00:00<?, ?it/s]

done in 17.02 seconds, 4.82 sentences/sec

Epoch 3 BERTScore
Precision: 0.903802
Recall:    0.895143
F1:        0.899249
F1 change vs baseline: +0.035380
Metrics saved: /kaggle/working/gpt_oss_20b_oncology_lora_stable_v2/epoch_bertscore_results.json
Predictions saved: /kaggle/working/gpt_oss_20b_oncology_lora_stable_v2/epoch_predictions/epoch_3_predictions.jsonl

Training completed successfully.
Final LoRA adapter: /kaggle/working/gpt_oss_20b_oncology_lora_stable_v2/final_adapter
Metrics: /kaggle/working/gpt_oss_20b_oncology_lora_stable_v2/epoch_bertscore_results.json
Predictions directory: /kaggle/working/gpt_oss_20b_oncology_lora_stable_v2/epoch_predictions
Final training metrics:
{'train_runtime': 14168.4486, 'train_samples_per_second': 0.537, 'train_steps_per_second': 0.067, 'total_flos': 1.5769214441496576e+17, 'train_loss': 1.2760048479903918, 'epoch': 3.0}
